In [ ]:
import json, re, subprocess, sys, zipfile, io
from collections import defaultdict
from pathlib import Path
import pandas as pd

def clone_repo(github_url, dest="cloned_repo"):
    dest_path = Path(dest)
    if dest_path.exists():
        print(f"{dest_path} already exists, skipping clone.")
        return dest_path
    result = subprocess.run(["git", "clone", "--depth", "1", github_url, str(dest_path)],
                             capture_output=True, text=True)
    if result.returncode != 0:
        print("CLONE FAILED:", result.stderr); sys.exit(1)
    print(f"Cloned {github_url} -> {dest_path}")
    return dest_path

repo_dir = clone_repo("https://github.com/your-org/your-service.git")

In [ ]:
def build_class_index_from_jar_folder(folder_path):
    """class -> ARTIFACT only (no group). Ground truth from real .class
    entries, no cross-referencing against declared dependencies, so
    transitively-used libraries are detected correctly too."""
    folder = Path(folder_path)
    jar_files = list(folder.glob("*.jar"))
    print(f"Found {len(jar_files)} jar file(s) in {folder}")
    class_index = {}
    for jar_path in jar_files:
        m = re.match(r"^(.+?)-(\d[\w.\-]*)\.jar$", jar_path.name)
        artifact = m.group(1) if m else jar_path.stem
        with zipfile.ZipFile(jar_path) as jar:
            for name in jar.namelist():
                if name.endswith(".class") and "/" in name and "module-info" not in name:
                    class_index[name[:-6].replace("/", ".")] = artifact
    return class_index

class_index = build_class_index_from_jar_folder("/path/to/your/local/jar/folder")

def build_class_index_from_boot_jar(boot_jar_path):
    class_index = {}
    with zipfile.ZipFile(boot_jar_path) as outer:
        lib_entries = [n for n in outer.namelist() if n.startswith("BOOT-INF/lib/") and n.endswith(".jar")]
        for entry_name in lib_entries:
            jar_filename = entry_name.split("/")[-1]
            inner_bytes = outer.read(entry_name)
            m = re.match(r"^(.+?)-(\d[\w.\-]*)\.jar$", jar_filename)
            artifact = m.group(1) if m else jar_filename.replace(".jar", "")
            with zipfile.ZipFile(io.BytesIO(inner_bytes)) as inner:
                for name in inner.namelist():
                    if name.endswith(".class") and "/" in name and "module-info" not in name:
                        class_index[name[:-6].replace("/", ".")] = artifact
    return class_index

# class_index = build_class_index_from_boot_jar("/path/to/your-service.jar")

In [ ]:
def load_category_map(path):
    df = pd.read_excel(path, sheet_name="library_categories")
    return dict(zip(df["artifact"], df["category"]))

CATEGORY = load_category_map("playbook.xlsx")  # sheet columns: artifact | category

In [ ]:
IMPORT_RE = re.compile(r"^\s*import\s+(?:static\s+)?([\w.]+)\s*;", re.MULTILINE)

def scan_source(repo_dir, class_index):
    files = [f for f in list(Path(repo_dir).rglob("*.java")) + list(Path(repo_dir).rglob("*.groovy"))
             + list(Path(repo_dir).rglob("*.kt")) if "/build/" not in str(f)]
    print(f"Scanning {len(files)} source file(s)")
    usage_count = defaultdict(int)
    for f in files:
        for imp in IMPORT_RE.findall(f.read_text()):
            artifact = class_index.get(imp)
            if artifact:
                usage_count[artifact] += 1
    return usage_count

usage_count = scan_source(repo_dir, class_index)
print(dict(usage_count))

In [ ]:
clusters = defaultdict(list)
for artifact in usage_count:
    clusters[CATEGORY.get(artifact, "uncategorized")].append(artifact)
duplicate_clusters = {c: v for c, v in clusters.items() if len(v) > 1}

print("DUPLICATE-FUNCTIONALITY CLUSTERS (for human review):")
for cat, artifacts in duplicate_clusters.items():
    print(f"  [{cat}] {artifacts}")

In [ ]:
def scan_and_report(repo_dir, class_index, CATEGORY):
    usage_count = scan_source(repo_dir, class_index)
    used = sorted(usage_count.keys())

    clusters = defaultdict(list)
    uncategorized = []

    for artifact in used:
        cat = CATEGORY.get(artifact)
        if cat is None:
            uncategorized.append(artifact)       # genuinely unknown purpose -- NOT a duplicate signal
        else:
            clusters[cat].append(artifact)         # has a known category -- CAN be checked for duplicates

    # Only a REAL category with 2+ libraries counts as a duplicate cluster.
    # "uncategorized" is explicitly excluded here -- it's a to-do list for
    # the playbook, not a finding about overlapping functionality.
    duplicate_clusters = {cat: artifacts for cat, artifacts in clusters.items() if len(artifacts) > 1}

    print(f"USED ({len(used)}): {used}")

    print(f"\nDUPLICATE-FUNCTIONALITY CLUSTERS ({len(duplicate_clusters)}) -- for human review:")
    if not duplicate_clusters:
        print("  none found")
    for cat, artifacts in duplicate_clusters.items():
        print(f"  [{cat}] {artifacts}")

    print(f"\nUNCATEGORIZED ({len(uncategorized)}) -- not duplicates, just missing from the playbook:")
    for a in uncategorized:
        print(f"  - {a}")

    return duplicate_clusters, uncategorized

duplicate_clusters, uncategorized = scan_and_report(repo_dir, class_index, CATEGORY)

In [ ]:
print("=== CATEGORY (from Excel) ===")
print("jackson-databind in CATEGORY?", "jackson-databind" in CATEGORY, "->", CATEGORY.get("jackson-databind"))
print("gson in CATEGORY?", "gson" in CATEGORY, "->", CATEGORY.get("gson"))
print("All CATEGORY keys (repr, to catch hidden whitespace):")
for k in CATEGORY.keys():
    print(f"  {repr(k)}")

print("\n=== used (from scan_source) ===")
print("jackson-databind in used?", "jackson-databind" in used)
print("gson in used?", "gson" in used)
print("All used artifacts (repr):")
for u in used:
    print(f"  {repr(u)}")

print("\n=== class_index (raw jar-derived data) ===")
print("Entries mapping to gson:", [k for k, v in class_index.items() if "gson" in v.lower()])
print("Entries mapping to jackson:", [k for k, v in class_index.items() if "jackson" in v.lower()])

In [ ]:
def load_category_map(path):
    df = pd.read_excel(path, sheet_name="library_categories")
    df["artifact"] = df["artifact"].astype(str).str.strip().str.lower()
    df["category"] = df["category"].astype(str).str.strip()
    return dict(zip(df["artifact"], df["category"]))

In [ ]:
cat = CATEGORY.get(artifact.strip().lower())

In [ ]:
print("jackson-databind category:", repr(CATEGORY.get("jackson-databind")))
print("gson category:           ", repr(CATEGORY.get("gson")))

In [ ]:
def load_playbook(path):
    """Loads BOTH sheets -- the raw category tags AND the per-category
    recommendation/reasoning -- so they can be displayed together."""
    cat_map_df = pd.read_excel(path, sheet_name="library_categories")
    CATEGORY = dict(zip(
        cat_map_df["artifact"].astype(str).str.strip().str.lower(),
        cat_map_df["category"].astype(str).str.strip()
    ))

    rec_df = pd.read_excel(path, sheet_name="categories")
    RECOMMENDATIONS = {
        row["category"]: {"recommended": row["recommended"], "reason": row["reason"]}
        for _, row in rec_df.iterrows()
    }
    return CATEGORY, RECOMMENDATIONS

CATEGORY, RECOMMENDATIONS = load_playbook("services_playbook.xlsx")

In [ ]:
def print_clusters_with_reason(duplicate_clusters, RECOMMENDATIONS):
    print(f"DUPLICATE-FUNCTIONALITY CLUSTERS ({len(duplicate_clusters)}) -- for human review:")
    for cat, artifacts in duplicate_clusters.items():
        print(f"\n[{cat}] {artifacts}")
        rec = RECOMMENDATIONS.get(cat)
        if rec:
            print(f"  recommended: {rec['recommended']}")
            print(f"  reason:      {rec['reason']}")
        else:
            print(f"  (no recommendation in playbook yet -- only the category tag exists, not the reasoning)")

print_clusters_with_reason(duplicate_clusters, RECOMMENDATIONS)